# AND ERA5-Land Annual Test

This notebook launches a focused annual ERA5-Land rerun for the Andrews (AND/HJA) watersheds. It uses the same extraction code as the main workflow and keeps only the AND watersheds before exporting.

This is the corrected small-watershed run: it calculates each value from the watershed polygon first, then retries the same polygon at a finer scale if the first result is blank. It does not fill blank values from the watershed centroid. Rows filled by the retry are marked with `used_fine_scale_fallback`.

The default run is 2001-2023 so it overlaps with the reference MODIS and climate-driver columns. It writes one CSV per year to the Google Drive folder printed below.


In [ ]:
REPO_URL = "https://github.com/global-river-chem/data-workflow_spatial.git"
REPO_DIR = "/content/data-workflow_spatial"

import os
import subprocess

# Clone the workflow the first time; refresh it on later notebook runs.
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)


In [ ]:
!pip -q install -r requirements.txt


In [ ]:
from src.gee_spatial.auth import initialize
from src.gee_spatial.extraction import load_run_config

assets = load_run_config("config/gee-assets.yml")
products = load_run_config("config/driver-products.yml")
run_list = load_run_config("config/run-list.yml")

DRIVE_EXPORT_FOLDER = "Google-Earth-Engine"
DRIVE_EXPORT_FOLDER_ID = "1Y4Hz9_vZsar61jjhYOrQXG4AR1oQWNAX"
DRIVE_EXPORT_FOLDER_URL = (
    "https://drive.google.com/drive/folders/"
    f"{DRIVE_EXPORT_FOLDER_ID}?usp=share_link"
)
# Point this run's Drive exports at the shared GEE output folder.
assets.setdefault("exports", {}).update(
    {
        "destination": "drive",
        "drive_folder": DRIVE_EXPORT_FOLDER,
        "drive_folder_id": DRIVE_EXPORT_FOLDER_ID,
        "drive_folder_url": DRIVE_EXPORT_FOLDER_URL,
    }
)

ee = initialize(project=assets.get("earth_engine", {}).get("project"))

print("Earth Engine project:", assets.get("earth_engine", {}).get("project"))
print("Watershed asset:", assets["watersheds"]["asset_id"])
print("Drive output folder:", assets["exports"].get("drive_folder_url"))


In [ ]:
from src.gee_spatial.extraction import load_watersheds
from src.gee_spatial.runs import choose_property_name

watersheds = load_watersheds(assets["watersheds"]["asset_id"])
property_names = watersheds.first().propertyNames().getInfo()

# Use the column names from the uploaded watershed asset, even if capitalization changed.
lter_column = choose_property_name(watersheds, "lter", alternatives=["LTER"])
site_id_column = choose_property_name(watersheds, "site_id")
lter_values = watersheds.aggregate_array(lter_column).distinct().sort().getInfo()
# Find the asset's exact spelling of AND before filtering rows.
and_lter_value = next((value for value in lter_values if str(value).strip().lower() == "and"), None)

print("Asset rows:", watersheds.size().getInfo())
print("Asset columns:", property_names)
print("LTER column:", lter_column)
print("Site ID column:", site_id_column)
print("LTER values:", lter_values)

# Stop before launching anything if the uploaded asset did not contain AND rows.
if and_lter_value is None:
    raise ValueError("No AND value was found in the uploaded watershed asset LTER column.")

# Keep only the Andrews rows for this focused check.
and_watersheds = watersheds.filter(ee.Filter.eq(lter_column, and_lter_value))
selected_row_count = and_watersheds.size().getInfo()

print("AND LTER value:", and_lter_value)
print("AND watershed rows:", selected_row_count)
print("AND preview:")
print(and_watersheds.limit(5).getInfo())

# Stop before launching anything if the AND filter did not keep any rows.
if selected_row_count == 0:
    raise ValueError("No AND watershed rows were selected.")


## Launch Annual Exports

This starts one Earth Engine table export per year. To try just a few years first, change `MAX_TASKS_TO_LAUNCH` to a small number like `3`.

The output file names include `AND_fine_scale` so they are easy to separate from the earlier centroid-fallback files. Only rerun this cell if you mean to start another set of exports.


In [ ]:
from pathlib import Path

from src.gee_spatial.export import export_table, task_summary
from src.gee_spatial.extraction import era5_export_columns, extract_era5_land_products
from src.gee_spatial.timing import (
    datetime_label,
    task_timing_row,
    timing_rows_from_launched_tasks,
    utc_now,
    write_timing_log,
)

START_YEAR = 2001
END_YEAR = 2023
MAX_TASKS_TO_LAUNCH = None
RUN_LABEL = "fine_scale"
PRODUCTS = [
    "precip",
    "temp",
    "evapotrans",
    "potential_evap",
    "snow_cover",
    "snow_water_equiv",
]

years_to_launch = list(range(START_YEAR, END_YEAR + 1))
# Leave MAX_TASKS_TO_LAUNCH as None for the full run, or set a small number for a test.
if MAX_TASKS_TO_LAUNCH is not None:
    years_to_launch = years_to_launch[: int(MAX_TASKS_TO_LAUNCH)]

session_started_at = utc_now()
timing_log_path = Path("timing-logs") / (
    f"gee_task_timing_era5_land_AND_{RUN_LABEL}_annual_{START_YEAR}_{END_YEAR}_"
    f"{datetime_label(session_started_at)}.csv"
)
timing_log_path.parent.mkdir(exist_ok=True)

selectors = era5_export_columns(products, product_names=PRODUCTS)
launched_tasks = []

print("Years to launch:", years_to_launch)
print("AND watershed rows:", selected_row_count)
print("Drive output folder:", assets["exports"].get("drive_folder_url"))
print("Export columns:", selectors)
print("Timing log:", timing_log_path)

# Launch one export per year so each CSV stays small and easy to rerun.
for year in years_to_launch:
    out_name = f"era5_land_{year}_AND_{RUN_LABEL}_watershed_extract"
    print("\nLaunching:", out_name)

    # Build the Earth Engine table for this year and this set of products.
    export_rows = extract_era5_land_products(
        products,
        and_watersheds,
        year=year,
        product_names=PRODUCTS,
    )

    # Start the Drive export. Earth Engine will run this in the background.
    task = export_table(
        export_rows,
        description=out_name,
        export_config=assets["exports"],
        file_name_prefix=out_name,
        selectors=selectors,
    )

    # Save enough timing information to check progress later.
    launched_at = utc_now()
    run_row = {
        "run_name": f"era5_land_AND_{RUN_LABEL}_annual_{START_YEAR}_{END_YEAR}",
        "mode": "era5_land",
        "products": PRODUCTS,
        "year": year,
        "month": None,
        "months": None,
        "period": "annual",
        "run_group": f"AND_{RUN_LABEL}",
    }
    timing_row = task_timing_row(
        run_row=run_row,
        export_name=out_name,
        task=task,
        selected_rows=selected_row_count,
        export_destination=assets["exports"].get("destination", "drive"),
        launched_at=launched_at,
    )
    launched_tasks.append(
        {
            "name": out_name,
            "task": task,
            "launched_at": launched_at,
            "timing_row": timing_row,
        }
    )
    write_timing_log(timing_rows_from_launched_tasks(launched_tasks), timing_log_path)
    print(task_summary(task))

print("\nLaunched tasks:", len(launched_tasks))
print("Timing log:", timing_log_path)


## Check Task Progress

Run this cell after the export cell. It checks Earth Engine once per minute until every task finishes or fails.


In [ ]:
import time

from src.gee_spatial.timing import (
    DONE_STATES,
    print_timing_summary,
    timing_rows_from_launched_tasks,
    update_task_timing_row,
    utc_now,
    write_timing_log,
)

poll_seconds = 60

# If this notebook was reopened, the launch list may not exist in memory.
if "launched_tasks" not in globals() or not launched_tasks:
    print("No tasks were launched in this session.")
else:
    # Keep checking until every Earth Engine task reaches a final state.
    while True:
        now = utc_now()
        print("\nTask status at", now.isoformat(timespec="seconds"))
        all_done = True

        # Refresh the status for each task launched in this notebook session.
        for item in launched_tasks:
            item["timing_row"] = update_task_timing_row(
                item["timing_row"],
                item["task"],
                checked_at=now,
            )
            state = item["timing_row"].get("state")
            elapsed_min = float(item["timing_row"].get("elapsed_min") or 0)
            print(f"{item['name']}: {state}, elapsed {elapsed_min:.1f} min")

            # Any unfinished task keeps the loop running.
            if state not in DONE_STATES:
                all_done = False

        write_timing_log(timing_rows_from_launched_tasks(launched_tasks), timing_log_path)

        # Once every task is done or failed, leave the checking loop.
        if all_done:
            break

        time.sleep(poll_seconds)

    print_timing_summary(timing_rows_from_launched_tasks(launched_tasks))
    print("Timing log:", timing_log_path)


## Place CSVs In The Shared Drive Folder

Earth Engine exports by Drive folder name. This cell checks the finished CSVs and moves them into the exact shared Drive folder ID from the link above. It only moves CSV outputs; it does not upload code.


In [ ]:
from google.colab import auth as colab_auth
from googleapiclient.discovery import build

colab_auth.authenticate_user()
drive_service = build("drive", "v3")

expected_csv_names = [
    f"era5_land_{year}_AND_{RUN_LABEL}_watershed_extract.csv"
    for year in years_to_launch
]

# Keep file names safe inside a Google Drive search query.
def drive_query_text(value):
    return value.replace("\\", "\\\\").replace("'", "\\'")


# Find visible Drive files with the exact CSV name. Newest matches come first.
def find_drive_files_by_name(file_name):
    query = f"name = '{drive_query_text(file_name)}' and trashed = false"
    response = drive_service.files().list(
        q=query,
        fields="files(id, name, parents, modifiedTime)",
        orderBy="modifiedTime desc",
        includeItemsFromAllDrives=True,
        supportsAllDrives=True,
    ).execute()
    return response.get("files", [])


moved_files = []
missing_files = []

# Move each finished CSV into the exact shared folder from the Drive link.
for file_name in expected_csv_names:
    matches = find_drive_files_by_name(file_name)

    # If Drive has not indexed the file yet, leave a clear list to check later.
    if not matches:
        missing_files.append(file_name)
        print("Not found yet:", file_name)
        continue

    # Prefer a file that is already in the shared folder; otherwise use the newest match.
    matches_in_target = [
        item for item in matches
        if DRIVE_EXPORT_FOLDER_ID in item.get("parents", [])
    ]
    drive_file = matches_in_target[0] if matches_in_target else matches[0]
    current_parents = drive_file.get("parents", [])

    # Nothing to move when the CSV is already in the shared folder.
    if DRIVE_EXPORT_FOLDER_ID in current_parents:
        print("Already in shared folder:", file_name)
        moved_files.append(file_name)
        continue

    update_args = {
        "fileId": drive_file["id"],
        "addParents": DRIVE_EXPORT_FOLDER_ID,
        "fields": "id, name, parents",
        "supportsAllDrives": True,
    }

    # Remove the old parent folder when Drive reports one, so the file appears only once.
    if current_parents:
        update_args["removeParents"] = ",".join(current_parents)

    drive_service.files().update(**update_args).execute()
    moved_files.append(file_name)
    print("Moved to shared folder:", file_name)

print("CSV files in shared folder:", len(moved_files))
print("CSV files not found yet:", len(missing_files))
if missing_files:
    print("Run this cell again in a few minutes for:")
    print("\n".join(missing_files))


## After The Exports Finish

Download the `era5_land_YYYY_AND_fine_scale_watershed_extract.csv` files from the shared Google Drive folder printed above. After they are in `Downloads`, or in `generated_outputs/and_era5_spatial_driver_comparison_YYYYMMDD/era5_land_AND_fine_scale_2001_2023`, rerun the local comparison script:

```bash
Rscript tools/plot_and_era5_modis_comparison.R
```
